SETTING UP THE API

In [1]:
from groq import Groq
import config

In [2]:
client = Groq(api_key=config.groq_api_key)

GENERATING TEXT

In [3]:
def generate_text(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
    ) 
    return response.choices[0].message.content

In [4]:
prompt = "Once upon a time."

In [5]:
generated_text = generate_text(prompt)
print(generated_text)

...in a far-off kingdom, where the sun dipped into the horizon and painted the sky with hues of crimson and gold. In this enchanted land, a magical forest whispered secrets to the wind, and a gentle river flowed with a voice that soothed the soul. It was here that our story began, with a young adventurer named... (Would you like me to continue the story, or would you like to give me some hints or directions?)


CUSTOMIZING THE OUTPUT

In [6]:
def generate_text(prompt, max_tokens, temperature):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        max_tokens = max_tokens,
        temperature = temperature
    ) 
    return response.choices[0].message.content

In [7]:
generated_text = generate_text(prompt, 25, 0)
print(generated_text)

...in a far-off kingdom, where the sun dipped into the horizon and painted the sky with hues of crimson and gold,


In [8]:
generated_text = generate_text(prompt, 25, 1)
print(generated_text)

It seems like you're about to start a classic fairy tale. Would you like to continue the story yourself, or would you


SUMMARIZING TEXT

In [9]:
def text_summarizer(prompt, max_tokens=256, temperature=0.5):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You will be provided with a block of text, and your task is to extract a list of keywords from it.",
            },
            {
                "role": "user",
                "content": 'A flying saucer seen by a guest house, a 7ft alien-like figure coming out of a hedge and a "cigar-shaped" UFO near a school yard.',
            },
            {
                "role": "assistant",
                "content": "flying saucer, guest house, 7ft alien-like figure, hedge, cigar-shaped UFO, school yard, extraterrestrial encounters, UK, mass sighting",
            },
            {
                "role": "user",
                "content": "Each April, in the village of Maeliya in northwest Sri Lanka, Pinchal Weldurelage Siriwardene gathers his community under the shade of a banyan tree near the wewa reservoir to prepare for the Sinhala New Year.",
            },
            {
                "role": "assistant",
                "content": "April, Maeliya, northwest Sri Lanka, Pinchal Weldurelage Siriwardene, banyan tree, wewa, reservoir, tank, Sinhala, rice paddies",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [10]:
prompt = "Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male mantaNormally I can help with things like this, but I don't seem to have access to that content. You can try again or ask me for something else."
print(prompt)

Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male mantaNormally I can help with things like this, but I don't seem to have access to that content. You can try again or ask me for something else.


In [11]:
text_summarizer(prompt)

'Here are the keywords I was able to extract:\n\nManta, snorkel, mask, reef, guide, Kirsty Whitman, marine life, ocean, diving'

POETIC CHATBOT

In [12]:
def poetic_chatbot(prompt, max_tokens=256, temperature=1.0):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a poetic chatbot."},
            {"role": "user", "content": "When was Google founded?"},
            {
                "role": "assistant",
                "content": "In the late '90s, a spark did ignite, Google emerged, a radiant light. By Larry and Sergey, in '98, it was born, a search engine...",
            },
            {
                "role": "user",
                "content": "Which country has the youngest president?",
            },
            {
                "role": "assistant",
                "content": "Ah, the pursuit of youth in politics, a theme we explore. In Austria, Sebastian Kurz did implore, at the age of 31, his journey...",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

In [13]:
prompt = "Who is Ronaldo?"
print(poetic_chatbot(prompt))

A name that echoes, across the land, Cristiano Ronaldo, a master of the ball's command. With feet of fire, and a heart of gold, he dances, weaves, and scores to tales untold. A Portuguese prince, of soccer's noble breed, his legend grows, as his feats proceed.


LANGCHAIN

In [21]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_groq import ChatGroq
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [22]:
url = "https://365datascience.com/upcoming-courses"
loader = WebBaseLoader(url)
raw_documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(documents, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1482.62it/s]


In [23]:
memory = ConversationBufferMemory(
    memory_key="chat_history", return_messages=True
)

llm = ChatGroq(
    groq_api_key=config.groq_api_key,
    model_name="llama-3.3-70b-versatile",
    temperature=0,
)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=vectorstore.as_retriever(), memory=memory
)

In [26]:
query = "What is the most famous course on the 365DataScience platform?"
result = qa.invoke({"question": query})

In [27]:
print(result["answer"])

The most famous or best-selling course on the 365DataScience platform is "Introduction to Data and Data Science" with Martin Ganchev and Iliya Valchanov, with a rating of 4.8/5 and 18,019 reviews.
